# **Text Vectorization II: Word Embeddings**

---
### Natural Language Processing — Master in Applied Artificial Intelligence

Author: Lorena Calvo-Bartolomé (lcalvo@pa.uc3m.es). Updated by Jerónimo Arenas-García (jarenas@ing.uc3m.es)

Version 2.0 (November 2026). First version: November 2022.

---
This notebook is based on the Gensim documentation and tutorials, as well as on Daniel Voigt Godoy's book "Deep Learning with PyTorch Step-by-Step, Volume III: Sequences & NLP".

Our goal here is to provide a basic overview of the following text vectorization techniques:

*   Word2Vec
*   GloVe
*   FastText
*   Contextual word embeddings

as well as how to use them to solve a text classification task, comparing the results with the sparse representations of Text Vectorization I.

In [ ]:
# Packages that are not preinstalled in Google Colab
%pip install --quiet gensim compress-fasttext

In [ ]:
# Common imports
import os
import gc
import numpy as np
import pandas as pd
from termcolor import colored
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression

# Figures plotted inside the notebook
%matplotlib inline
# High quality figures
%config InlineBackend.figure_format = 'retina'
# Figures size
plt.rcParams['figure.figsize'] = [8, 6]

# Suppress future warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', module='compress_fasttext')

In [ ]:
# Colab-specific helpers (they are skipped when running the notebook elsewhere)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Fancy (interactive) display of dataframes
    get_ipython().run_line_magic('load_ext', 'google.colab.data_table')

# To wrap long text lines in outputs
from IPython.display import HTML, display
def set_css(*args, **kwargs):
    display(HTML('<style> pre { white-space: pre-wrap; } </style>'))
get_ipython().events.register('pre_run_cell', set_css)

### Data files

This notebook uses two files generated by the Text Vectorization I notebook: `corpus_df_imdb.csv` and `imdb_lemmas_clean.txt`. If you did not finish that notebook, you can find both files on Aula Global. Copy them to the folder you used for Text Vectorization I and update `DATA_DIR` accordingly.

In [ ]:
DATA_DIR = 'NLP_data'   # folder with the course data files when running locally

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/My Drive/NLP_data'  # UPDATE THIS ACCORDING TO WHERE YOU STORED THE FILES!!!!

print(os.listdir(DATA_DIR))

## **1. Data preparation**
---


For this second text vectorization notebook, we will keep working with the same 10,000 reviews from the **IMDB Dataset of 50K Movie Reviews**. Since we already carried out all the preprocessing in the previous notebook, we just need to load the dataframe that we saved. Column `clean_review` contains the clean lemmas of each review, separated by blank spaces.

In [ ]:
corpus_df = pd.read_csv(os.path.join(DATA_DIR, 'corpus_df_imdb.csv'))
corpus_df['clean_review'] = corpus_df['clean_review'].fillna('')
print(len(corpus_df))
corpus_df.head()

##### **Exercise 1**

Create a Gensim corpus (a list of lists of tokens) and obtain its Gensim dictionary. With the latter, save the vocabulary of the corpus as a list. Store the Gensim corpus in a variable named `corpus`, the Gensim dictionary in a variable named `D`, and the vocabulary in a variable named `vocab`.

**Hint:** Make use of the Gensim dictionary attribute `token2id` for obtaining the vocabulary list.

In [ ]:
from gensim.corpora import Dictionary

# <SOL>

# </SOL>

print(colored('\n============= First 20 words in the vocabulary =============', 'blue'))
print(vocab[0:20])
print(len(vocab))

Let's also save the sentiment of the review in a NumPy array for later use.

In [ ]:
Y = corpus_df['binary_sentiment'].values

## **2. Word2Vec**
---


We will use [Gensim's implementation of Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html), which supports both the skip-gram and CBOW models, with either hierarchical softmax or negative sampling. Furthermore, Gensim contains a collection of functions for the exploitation of pre-trained embeddings, which can be either embeddings developed by us or pre-trained models on big corpora.

### *2.1. Implementation of Word2Vec in Gensim*

Word2Vec models require a lot of text, e.g., the entire Wikipedia corpus. Nevertheless, we will demonstrate the principles using a small example from our IMDB dataset.

Gensim's Word2Vec expects a sequence of sentences (``sentences``) as its input where each sentence is a list of words:

```
sentences = [['This', 'is', 'the', 'first', 'sentence'], ['This', 'is', 'the', 'second', 'sentence']]
```

In addition, some of the most important parameters that we may wish to configure are the following:

* `vector_size`: The dimensionality of the word vectors, i.e.,  the embedding size. The default is **100**.
* `window`: The window size, i.e., the number of adjacent words that are considered in the same context as the target word. The default is **5**.
* `min_count`: minimum number of times a word must appear in the corpus to be considered in the model. The default is **5**.
* `sg`: Training algorithm: 1 for skip-gram; otherwise CBOW.

Keeping the input as a Python built-in list is convenient, but can use up a lot of RAM when the input is large. Gensim only requires the input to provide sentences sequentially, when iterated over. Hence, we can use the same principle that we utilized in the first notebook of this tutorial at ``2.6. Memory efficient computation``.

##### **Exercise 2**

The provided class `IterableSentence_fromfile` creates an iterator that yields one sentence (list of words) after another from the file `"imdb_lemmas_clean.txt"`.

Use the sentence iterator to create a Word2Vec skip-gram model named `model_w2v` with vector size **200** and window size **5**. Words must appear at least **10** times in the corpus to be kept in the model. Use a seed of **42**.

In [ ]:
%%time
from gensim.models import Word2Vec

class IterableSentence_fromfile(object):
    def __init__(self, filename):
        self.__filename = filename

    def __iter__(self):
        for line in open(self.__filename, encoding='utf-8'):
            # assume there's one sentence per line, tokens separated by whitespace
            yield line.split()

sentences = IterableSentence_fromfile(os.path.join(DATA_DIR, "imdb_lemmas_clean.txt")) # a memory-friendly iterator

# <SOL>

# </SOL>

Once the model is trained, it is accessible via the ``wv`` attribute, which is the actual word vector model in which queries can be made.

We can see the learned vocabulary of the model through the dictionary ``key_to_index``, that is, the target words for which we have calculated an embedding, as well as the corresponding embeddings for these words, as follows:

In [ ]:
print(colored('\n============= Word2Vec vocabulary =============', 'blue'))
words = list(model_w2v.wv.key_to_index)
print(words[0:50])
print(colored('\n============= Embedding of the first word =============', 'blue'))
print(model_w2v.wv[words[0]])

We can save the trained model by calling the ``save()`` function, and we can load it again by calling the ``load()`` method.

In [ ]:
model_w2v.save("model_w2v_imdb.model")

#model_w2v = Word2Vec.load("model_w2v_imdb.model")

Since we are not going to keep with the training at this moment, we do not need the full model state anymore. Hence, we can store the words and their trained embeddings and load them back as ``KeyedVectors``, which is essentially a mapping between keys and vectors. This results in a much smaller and faster object that can be mapped for lightning-fast loading and sharing the vectors in RAM between processes.

In [ ]:
from gensim.models import KeyedVectors

# Store just the words + their trained embeddings.
word_vectors = model_w2v.wv
word_vectors.save("model_w2v_imdb.wordvectors")

# Load back with memory-mapping = read-only, shared across processes.
wv = KeyedVectors.load("model_w2v_imdb.wordvectors", mmap='r')

Let's free the memory.



In [ ]:
del model_w2v
gc.collect()

Since we have represented our words as numerical vectors, we can utilize the Euclidean distance (or cosine similarity) among two word vectors to measure the linguistic or semantic similarity of the corresponding words. To do this, Gensim provides the function ``most_similar`` which returns the most similar words from a given word.

In [ ]:
wv.most_similar(positive="cinema")

In [ ]:
wv.most_similar(positive="restaurant")

We can also check the performance of our model by visualizing the embeddings. However, although embeddings are low-dimensional vectors, even 4 dimensions are too high to visualize. [t-SNE](https://en.wikipedia.org/wiki/T-distributed_stochastic_neighbor_embedding) (*t-distributed Stochastic Neighbor Embedding*) solves this problem as it allows for visualizing high-dimensional data by reducing them to two or three-dimensional data. The technique takes in the embeddings (or any data) and aims at preserving as much as possible in 2 (or 3) dimensions the distances from the original space of the embedding dimension. This, therefore, helps us to get a feel for the space of word embedding.


Sklearn includes an implementation of [t-SNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) that we can easily use.

Computing t-SNE for the whole vocabulary takes a long time, so we will use only the 500 most frequent words.

In [ ]:
from sklearn.manifold import TSNE

n_words = 500    # Words in the vocabulary are sorted by frequency
words_tsne = wv.index_to_key[:n_words]
embed_tsne = TSNE(init='random', random_state=42).fit_transform(np.asarray(wv.vectors[:n_words]))

fig, ax = plt.subplots(figsize=(16, 16))
for idx, word in enumerate(words_tsne):
    plt.scatter(*embed_tsne[idx, :], color='steelblue')
    plt.annotate(word, (embed_tsne[idx, 0], embed_tsne[idx, 1]), alpha=0.7)
plt.grid()

From the above results, we can see that our word2vec model does not do a bad job, but it could still be improved. Certainly, we need to train on a much larger dataset to obtain a coherent embedding structure.

### *2.2. Sentiment Analysis with ad-hoc trained Word2Vec embeddings*

Since our data contains reviews and not just words, we need to figure out a way to use the word vectors from the word2vec model to create a vector representation for an entire review. One simple solution to achieve this is to take the mean of all the word vectors present in the review.

Since when we constructed our Word2Vec model we filtered out those words which did not have an overall frequency of 10, we may encounter that our reviews contain some words that are not in the vocabulary of the model, i.e., they are out-of-vocabulary words (OOV).

##### **Exercise 3**

Implement the function ``get_vocabulary_coverage`` which, given a ``KeyedVectors`` object and the Gensim dictionary associated with the reviews, calculates the vocabulary coverage of the model. Take into account the frequency with which the OOV words appear in the reviews.

Then, calculate the coverage of the Word2Vec model on the IMDB vocabulary.

In [ ]:
def get_vocabulary_coverage(model, gensim_dict):
    """Fraction of the token occurrences in the corpus covered by the vocabulary of the embedding model"""
    #<SOL>

    #</SOL>
    return coverage

#<SOL>

#</SOL>

##### **Exercise 4**

Implement the function `get_review_vector` which, given a `KeyedVectors` object and a list of tokens representing a review, returns the embedding vector for such a review by taking the average of the vectors of the words present in the given review. Ignore the OOV words (and return a vector of zeros if all the words are OOV).

Once you have the function, calculate the embedding vectors of all the reviews and save them in a NumPy array `X_w2v` of dimensions $[n_{reviews}, e_{size}]$, where $e_{size}$ is the embedding size of the model.

In [ ]:
def get_review_vector(model, review):
    #<SOL>

    #</SOL>
    return vec

#<SOL>

#</SOL>
print(X_w2v.shape)

##### **Exercise 5**

Classify the reviews based on the Word2Vec embeddings with the same protocol as in Text Vectorization I: logistic regression, $70:30$ split with `random_state=42`, and `GridSearchCV` (5 folds) to choose `C` among `[0.01, 0.1, 1, 10]`. Evaluate the accuracy on the test set. We provide the function `evaluate_representation`, which you can reuse in the rest of the notebook.

How does the accuracy compare with the BoW and TF-IDF representations of Text Vectorization I?

In [ ]:
def evaluate_representation(X, Y, name):
    """Logistic regression with the same protocol as in Text Vectorization I (70/30 split, 5-fold CV for C)"""
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)
    grid = GridSearchCV(LogisticRegression(max_iter=2000), param_grid={'C': [0.01, 0.1, 1, 10]}, cv=5, n_jobs=-1)
    grid.fit(X_train, y_train)
    acc = grid.score(X_test, y_test)
    print(f'{name}: best C = {grid.best_params_["C"]}, test accuracy = {acc:.3f}')
    return acc

# <SOL>

# </SOL>

## **3. GloVe**

### *3.1. Pre-trained GloVe models in Gensim*

For a specific NLP task, training our own word vectors may be the optimal strategy. However, it may require a long time, a fast computer with plenty of RAM and disc space, and maybe some experience in fine-tuning the input data and training algorithm. Another option is to just utilize pre-trained word embeddings.

For example, Gensim includes functions to exploit a set of [pre-trained *embeddings*](https://github.com/RaRe-Technologies/gensim-data).

In [ ]:
from gensim import downloader

print(colored('\n============= Available pre-trained models in Gensim =============', 'blue'))
print(list(downloader.info()['models'].keys()))

As we can see above, many of the pretrained models provided by Gensim are GloVe models. The pretrained GloVe embeddings come in many sizes and shapes, with dimensions in the range $[25,300]$, and vocabularies varying between $400,000$ and $2,200,000$ words.

Let's use Gensim's downloader to get the ``glove-wiki-gigaword-50``, which is the smallest available GloVe model. It was trained on Wikipedia 2014 and Gigawords, contains $400,000$ vocabulary words and its embeddings have a dimension of $50$.

In [ ]:
glove = downloader.load('glove-wiki-gigaword-50')

print(f"The dictionary size is {glove.vectors.shape[0]}.\n\
        Each embedding has a dimension of {glove.vectors.shape[1]}")

In [ ]:
glove.most_similar(["restaurant"])

In [ ]:
type(glove)

### *3.2. Semantic operations with embeddings*

One interesting thing that we can do with word vectors, which is easily perceptible with the GloVe-based embeddings we just loaded, is to perform algebra arithmetic with words. A famous example is the following equation:

``
queen = (king - man) + woman
``

That is, the word ``queen`` is the closest word to the operation consisting of subtracting ``man`` from ``king`` and adding the word ``woman``. The quality of "being a man" is replaced with that of "being a woman", leading to the word queen.

We can perform this operation in Gensim as follows:

In [ ]:
glove.most_similar(positive=['woman', 'king'], negative=['man'], topn=1)

In [ ]:
synthetic_queen = glove['king'] - glove['man'] + glove['woman']
glove.similar_by_vector(synthetic_queen, topn=5)

The above result indicates that the term most comparable to the arithmetic operation result (synthetic queen) is the original word itself. This is pretty typical when conducting word arithmetic operations, which is why we usually omit the original word from the output, thus being the most similar word to our synthetic_queen the actual queen.

The general idea here is that the embeddings have learned to **encode abstract dimensions**, like gender, genealogy, or profession. Yet, none of these abstract dimensions correspond to a single numerical dimension. In its 50-dimensional feature space, the model learned to place ``the man`` as far apart from ``woman`` as ``king`` is from ``queen`` (roughly approximating the gender difference between the two). Similarly, the model learned to place ``king`` as far apart from ``man`` as ``queen`` is from ``woman`` (roughly approximating the difference of being a royal). We can visualize this concept when projected in two dimensions as shown below:

<img src="https://drive.google.com/uc?id=1HhXF9Emxk6kZRmLSpg1kLd1URudlcyjx" width="40%">

From the figure above, it should be relatively clear that both arrows pointing up (blue) are approximately the same size, thus resulting in the equation below:

\begin{align}
    w_{king} - w_{man} &≃ w_{queen} - w_{woman}\\
    w_{king} - w_{man} &+ w_{woman} ≃ w_{queen}
\end{align}

We can observe how the word vectors also include information to relate countries with nationalities, months of the year, family relationships, etc.

In [ ]:
result = glove.most_similar(positive=['woman', 'king'], negative=['man'], topn=1)
print('King - Man + Woman = ',result)
result = glove.most_similar(positive=['rome', 'france'], negative=['paris'], topn=1)
print('France - Paris + Rome = ',result)
result = glove.most_similar(positive=['english', 'france'], negative=['french'], topn=1)
print('France - french + english = ',result)
result = glove.most_similar(positive=['june', 'december'], negative=['november'], topn=1)
print('December - November + June = ',result)
result = glove.most_similar(positive=['sister', 'man'], negative=['woman'], topn=1)
print('Man - Woman + Sister = ',result)

But we do not always get the expected result...

In [ ]:
result = glove.most_similar(positive=['aunt', 'nephew'], negative=['niece'], topn=1)
print('nephew - niece + aunt = ',result)

In conclusion, word embeddings are indeed quite capable of capturing and maintaining the semantic relationship between different words.

### *3.3. Sentiment Analysis with pretrained GloVe embeddings*

##### **Exercise 6**

Analyze the vocabulary coverage of the GloVe embeddings on the IMBD dataset. How is it in comparison to the provided Word2Vec model trained on the IMBD corpus?

In [ ]:
# <SOL>

# </SOL>

##### **Exercise 7**

Obtain the embedding representation of the reviews based on GloVe's word vectors, train a classifier with them and evaluate its accuracy. Use the same configuration as in Exercise 5.

In [ ]:
# <SOL>

# </SOL>

In [ ]:
del glove
gc.collect()

## **4. FastText**

### *4.1. Representation of OOV with ad-hoc vs pre-trained fastText model in Gensim*

We will be using [Gensim's implementation to learn word representations via fastText](https://radimrehurek.com/gensim/models/fasttext.html), which allows training word embeddings from a training corpus with the additional ability to obtain word vectors for out-of-vocabulary words.

Similar to what we did with Word2Vec, we can obtain the ad-hoc embeddings of our reviews based on the fastText model. Input must be given in the same format as Word2Vec, i.e., as a sequence of sentences, where each sentence is a list of words, that can be given as an iterator for efficient memory management.

Many of the construction parameters are common to the Word2Vec class:

* `vector_size` (embedding size): the number of dimensions for the vector representation. The default is 100.
* `window` (window size): the number of adjacent words that are considered to be in the same context as a target word. The default is 5.
* `min_count`: minimum number of times a word must appear in the corpus to be considered in the model.

Once trained, we can similarly save/load the model as we saw in Word2Vec.

##### **Exercise 8**

Train a FastText model named `model_fasttext` on the IMDB dataset. Use an embedding size of $200$, a window size of $5$, words must appear at least $10$ times in the corpus to be kept in the model, and use a seed of $42$. Use the class `IterableSentence_fromfile` defined above to create an iterator that yields one sentence after another from the file `"imdb_lemmas_clean.txt"`.

In [ ]:
%%time
from gensim.models import FastText

# <SOL>

# </SOL>

In [ ]:
from gensim.models import KeyedVectors

# Save model
model_fasttext.save("model_fastText.model")

# Store just the words + their trained embeddings.
word_vectors = model_fasttext.wv
word_vectors.save("model_fastText.wordvectors")

# Load back with memory-mapping = read-only, shared across processes.
fastText_wv = KeyedVectors.load("model_fastText.wordvectors", mmap='r')

In [ ]:
del model_fasttext
gc.collect()

By representing the TSNE representation of the fastText embeddings we can see its better ability to group words with the same root (and similar meaning) as opposed to Word2Vec.

In [ ]:
words_tsne = fastText_wv.index_to_key[:n_words]
embed_tsne = TSNE(init='random', random_state=42).fit_transform(np.asarray(fastText_wv.vectors[:n_words]))

fig, ax = plt.subplots(figsize=(16, 16))
for idx, word in enumerate(words_tsne):
    plt.scatter(*embed_tsne[idx, :], color='steelblue')
    plt.annotate(word, (embed_tsne[idx, 0], embed_tsne[idx, 1]), alpha=0.7)
plt.grid()

In [ ]:
del embed_tsne
gc.collect()

Another interesting thing to look at from the fastText model is its ability to group words with the same root (and similar meaning). This allows fastText's ability to obtain embeddings for out-of-dictionary (OOV) words, as opposed to Word2Vec:

In [ ]:
# Word2Vec for word 'dichlorodiphenyltrichloroethane'
word = 'dichlorodiphenyltrichloroethane'
if (word in wv.key_to_index):
    print(wv[word])
else:
    print(f'The word {word} does not belong to the dictionary. Word2Vec cannot assign an embedding.')

In [ ]:
# fastText constructs an embedding from the embeddings of the constituent n-grams
if not (word in fastText_wv.key_to_index):
    print(f'The word {word} does not belong to the dictionary. The embedding constructed from the n-grams is:\n')
    print(fastText_wv[word])
    print('\nAnd its closest neighbors are:')
    print(fastText_wv.most_similar(word))

The OOV embedding neighbours make little sense since the embedding dictionary was learned on a relatively tiny dataset. Let us now consider the usage of **pre-trained models**.

Since we want to keep fastText ability to obtain the embedding representation of OOV, using Gensim's KeyedVectors is not an option, as it only maintains a simple word (as key) to vector (as value) mapping. To solve this issue, we are instead going to be using the library [Compress-fastText
](https://pypi.org/project/compress-fasttext/), that allows us to load pretrained fastText models as instances of the class **CompressedFastTextKeyedVectors**, which does synthesize vectors for out-of-vocabulary words.


In [ ]:
try:
  import compress_fasttext
except ModuleNotFoundError:
  %pip install compress-fasttext
  import compress_fasttext

In [ ]:
fastTextPre = compress_fasttext.models.CompressedFastTextKeyedVectors.load(
    'https://github.com/avidale/compress-fasttext/releases/download/v0.0.4/cc.en.300.compressed.bin'
)

In [ ]:
if not (word in fastTextPre.key_to_index):
    print(f'The word {word} does not belong to the dictionary. The embedding constructed from the n-grams is:\n')
    print(fastTextPre[word])
    print('\nAnd its closest neighbors are:')
    print(fastTextPre.most_similar(word))

As we can observe now, the OOV embedding neighbours do make sense.

### *4.2. Sentiment Analysis with ad-hoc and pre-trained fastText embeddings*

##### **Exercise 9**

Obtain the embedding representation of the reviews based on the **ad-hoc fastText word vectors obtained from the IMDB dataset**, train a classifier with them, and evaluate its performance. Use the same configuration as in Exercise 5.

In [ ]:
# <SOL>

# </SOL>

##### **Exercise 10**

Obtain the embedding representation of the reviews based on the **pre-trained fastText word vectors**, train a classifier with them, and evaluate its performance. Use the same configuration as in Exercise 5.

**Note:** the pre-trained model can build a vector for any word, so there are no OOV words in this case.

In [ ]:
# <SOL>

# </SOL>

Additionally, the Compress-fastText library allows feeding fastText embeddings to scikit-learn models through the `FastTextTransformer` class, which **represents a text as the average of the embeddings of its words**. Hence, we can directly build a pipeline with our classifier on top of fastText, as follows:

In [ ]:
from sklearn.pipeline import make_pipeline
from compress_fasttext.feature_extraction import FastTextTransformer

corpus_train, corpus_test, y_train, y_test = train_test_split(corpus_df['clean_review'].tolist(), Y, test_size=0.3, random_state=42)

classifier = make_pipeline(
    FastTextTransformer(model=fastTextPre),
    GridSearchCV(LogisticRegression(max_iter=2000), param_grid={'C': [0.01, 0.1, 1, 10]}, cv=5, n_jobs=-1)
).fit(corpus_train, y_train)
score_fastText_pipeline = classifier.score(corpus_test, y_test)
print("Accuracy with pre-trained fastText embeddings via FastTextTransformer:", score_fastText_pipeline)

In [ ]:
del fastTextPre
gc.collect()

##### **Exercise 11**

Summarize the accuracies obtained in this notebook in a table, together with the BoW and TF-IDF results of Text Vectorization I. Are dense representations obtained by averaging word embeddings better than sparse representations for this task? Why do you think so?

In [ ]:
# <SOL>

# </SOL>

## **5. Contextual word embeddings**

All the embeddings above are **static**: each word has a single vector, regardless of its context. **Contextual** word embeddings assign a different vector to each occurrence of a word, depending on the sentence in which it appears. ELMo, which we saw in the lecture, obtains them from the hidden states of a pre-trained bidirectional LSTM language model; today they are usually obtained from Transformer-based models, which we will study next week.

<font color='green'>The library previously used in this notebook to obtain ELMo embeddings (AllenNLP) has been archived. We use instead a BERT model from Hugging Face, which is preinstalled in Google Colab. You do not need to understand the model yet: just note that it returns one vector per token, which depends on the whole sentence.</font>

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert = AutoModel.from_pretrained('bert-base-uncased')
bert.eval()

def contextual_vector(sentence, word):
    """Contextual embedding (last hidden layer of BERT) of the first occurrence of `word` in `sentence`"""
    enc = tokenizer(sentence, return_tensors='pt')
    with torch.no_grad():
        hidden = bert(**enc).last_hidden_state[0]
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    return hidden[tokens.index(word)].numpy()

In [ ]:
sentences_bank = ["My investment adviser at my local bank called to discuss my portfolio.",
                  "I went to the bank to deposit my salary.",
                  "Every day this sweet old lady sits on the same bank at my local park to feed the ducks.",
                  "We had a picnic on the bank of the river."]

vectors_bank = np.array([contextual_vector(s, 'bank') for s in sentences_bank])
norms = np.linalg.norm(vectors_bank, axis=1, keepdims=True)
cos_sim = (vectors_bank / norms) @ (vectors_bank / norms).T

print(colored('Cosine similarity between the contextual embeddings of "bank":', 'blue'))
print(pd.DataFrame(cos_sim, index=[f'S{i+1}' for i in range(4)], columns=[f'S{i+1}' for i in range(4)]).round(2))

The two financial uses of *bank* (S1, S2) are more similar to each other than to the uses meaning *bench* or *river bank* (S3, S4). With Word2Vec, GloVe or fastText, the four occurrences would have exactly the same vector.

---
In this notebook, we have trained Word2Vec and fastText embeddings on our corpus, used pre-trained GloVe and fastText embeddings, and represented each review as the average of its word vectors. Note that averaging word vectors is a simple way to obtain document representations, but it loses a lot of information. In the next sessions we will see Transformer models, which provide contextual representations and better ways to represent sentences and documents (sentence embeddings), which you will use for retrieval in Milestone 2.